# Topología Aplicada y Computacional

## Práctica 3

Para el desarrollo de esta práctica se hará uso de los módulos `Simplex`, `SimplicialComplex`, `FilteredSimplicialComplex` y `AlphaFiltration` definidos en los laboratorios anteriores.

In [58]:
from homology.simplex import Simplex
from homology.simplicial_complex import SimplicialComplex
from homology.filtered_simplicial_complex import FilteredSimplicialComplex
from homology.filtrations.alpha_filtration import AlphaFiltration

**Exercise 1:** Definir un método dentro de la clase complejos simpliciales que calcule para cada p la matriz borde $M_{\partial_p}$.

En lugar de crear un método en la clase `SimplicialComplex`, se ha creado la clase *wrapper* `MatrixSimplicialComplex`, que se inicializa con un complejo simplical. Esto se hace tanto para seguir el principio de *separation of concerns* como para poder acelerar sus distintas funciones usando cachés.

Una vez provista de un complejo simplicial, la clase `SimplicialComplex` inicializa dos cachés:

- `_n_faces`: guarda los símplices de las llamadas a `SimplicialComplex.n_faces`, para no tener que recomputarlos.
- `_n_delta`: guarda las matrices de las llamadas a `MatrixSimplicialComplex.boundary_matrix`, para no tener que recomputarlas.

De este modo, el método `n_faces` busca las caras de dimensión $n$ en la caché y si no las encuentra las computa con el método de `SimplicialComplex`, y `boundary_matrix` busca la matriz de borde de la dimensión $n$ dada en la caché y si no la encuentra la computa usando los símplices de dimensión $n$ y $n-1$ del complejo simplicial.

In [59]:
from __future__ import annotations

import numpy as np

class MatrixSimplicialComplex:
    """Compute boundary matrices and Betti numbers over GF(2)."""

    def __init__(self, sc: SimplicialComplex) -> None:
        """Initialize the complex and its caches."""
        self._sc: SimplicialComplex = sc

        # The wrapped complex is assumed not to change after construction.
        self._n_faces: dict[int, list[Simplex]] = {}
        self._n_delta: dict[int, np.ndarray] = {}
        
    def n_faces(self, dim: int) -> list[Simplex]:
        """Return the simplices of the given dimension."""
        if dim < -1:
            raise ValueError("Dimension must be at least -1")
        if dim == -1 or dim > self._sc.dim:
            return []

        if dim not in self._n_faces:
            self._n_faces[dim] = sorted(
                self._sc.n_faces(dim),
                key=lambda simplex: simplex.vertices,
            )

        return self._n_faces[dim]

    def boundary_matrix(self, dim: int) -> np.ndarray:
        """Return the mod-2 boundary matrix from dimension dim to dim - 1."""
        if dim not in self._n_delta:
            row_faces = self.n_faces(dim - 1)
            col_faces = self.n_faces(dim)

            mat = np.zeros((len(row_faces), len(col_faces)), dtype=np.uint8)

            # Rows are (dim - 1)-faces and columns are dim-faces.
            for i, row_face in enumerate(row_faces):
                for j, col_face in enumerate(col_faces):
                    if row_face.is_face(col_face):
                        mat[i, j] = 1

            self._n_delta[dim] = mat

        return self._n_delta[dim]

Comprobamos el funcionamiento de la función calculando la matriz borde de dimensión 2 de un tetraedro:

In [60]:
sc = SimplicialComplex([Simplex((1,2,3,4))])
MatrixSimplicialComplex(sc).boundary_matrix(2)

array([[1, 1, 0, 0],
       [1, 0, 1, 0],
       [0, 1, 1, 0],
       [1, 0, 0, 1],
       [0, 1, 0, 1],
       [0, 0, 1, 1]], dtype=uint8)

**Exercise 2:** Definir una función que calcule la forma normal de Smith de una matriz con coeficientes en $\mathbb{Z}_2$.

Como la forma normal de Smith en $\mathbb{Z}_2$ está completamente determinada por el rango de su matriz de borde correspondiente, se crea una nueva caché `_n_rank`. De este modo, cada llamada a `smith_boundary_matrix` comprueba si se conoce el rango de la matriz. Si se conoce, se reconstruye la matriz de forma eficiente. En caso contrario, se computa su rango usando un algoritmo de eliminación de filas y columnas.

In [61]:
class MatrixSimplicialComplex:
    """Compute boundary matrices and Betti numbers over GF(2)."""

    def __init__(self, sc: SimplicialComplex) -> None:
        """Initialize the complex and its caches."""
        self._sc: SimplicialComplex = sc

        # The wrapped complex is assumed not to change after construction.
        self._n_faces: dict[int, list[Simplex]] = {}
        self._n_delta: dict[int, np.ndarray] = {}
        self._n_rank: dict[int, int] = {}

    def n_faces(self, dim: int) -> list[Simplex]:
        """Return the simplices of the given dimension."""
        if dim < -1:
            raise ValueError("Dimension must be at least -1")
        if dim == -1 or dim > self._sc.dim:
            return []

        if dim not in self._n_faces:
            self._n_faces[dim] = sorted(
                self._sc.n_faces(dim),
                key=lambda simplex: simplex.vertices,
            )

        return self._n_faces[dim]

    def boundary_matrix(self, dim: int) -> np.ndarray:
        """Return the mod-2 boundary matrix from dimension dim to dim - 1."""
        if dim not in self._n_delta:
            row_faces = self.n_faces(dim - 1)
            col_faces = self.n_faces(dim)

            mat = np.zeros((len(row_faces), len(col_faces)), dtype=np.uint8)

            # Rows are (dim - 1)-faces and columns are dim-faces.
            for i, row_face in enumerate(row_faces):
                for j, col_face in enumerate(col_faces):
                    if row_face.is_face(col_face):
                        mat[i, j] = 1

            self._n_delta[dim] = mat

        return self._n_delta[dim]

    def rank_boundary_matrix(self, dim: int) -> int:
        """Return the rank of the boundary matrix over GF(2)."""
        if dim not in self._n_rank:
            mat = self.boundary_matrix(dim).copy()
            rows, cols = mat.shape

            rank = min(rows, cols)

            for k in range(min(rows, cols)):
                # Find a pivot in the remaining submatrix.
                nz = np.argwhere(mat[k:, k:] != 0)

                if nz.size == 0:
                    rank = k
                    break

                # Move the pivot to (k, k).
                i, j = nz[0] + k

                mat[[k, i], :] = mat[[i, k], :]
                mat[:, [k, j]] = mat[:, [j, k]]

                # Clear the rest of the pivot row and column.
                for nzcol in np.flatnonzero(mat[k, k+1:]) + k+1:
                    mat[:, nzcol] ^= mat[:, k]

                for nzrow in np.flatnonzero(mat[k+1:, k]) + k+1:
                    mat[nzrow, :] ^= mat[k, :]

            self._n_rank[dim] = rank

        return self._n_rank[dim]

    def smith_boundary_matrix(self, dim: int) -> np.ndarray:
        """Return the Smith normal form over GF(2)."""
        rank = self.rank_boundary_matrix(dim)
        mat = np.zeros(self.boundary_matrix(dim).shape, dtype=np.uint8)

        for k in range(rank):
            mat[k, k] = 1

        return mat

Comprobamos el funcionamiento de la función calculando la forma normal de Smith de la matriz anterior:

In [62]:
sc = SimplicialComplex([Simplex((1,2,3,4))])
msc = MatrixSimplicialComplex(sc)
msc.smith_boundary_matrix(2)

array([[1, 0, 0, 0],
       [0, 1, 0, 0],
       [0, 0, 1, 0],
       [0, 0, 0, 0],
       [0, 0, 0, 0],
       [0, 0, 0, 0]], dtype=uint8)

**Exercise 3:** Definir un método que calcule los números de Betti de un complejo simplicial.

Una vez completados los dos ejercicios anteriores, obtener el número de Betti es trivial. Como $\beta_p(\sigma) = Z_p(\sigma) - B_p(\sigma)$, basta con obtener las matrices de borde en forma normal de Smith de dimensión $p$ y dimensión $p+1$ para conseguir $Z_p(\sigma)$ y $B_p(\sigma)$, respectivamente.

In [63]:
class MatrixSimplicialComplex:
    """Compute boundary matrices and Betti numbers over GF(2)."""

    def __init__(self, sc: SimplicialComplex) -> None:
        """Initialize the complex and its caches."""
        self._sc: SimplicialComplex = sc

        # The wrapped complex is assumed not to change after construction.
        self._n_faces: dict[int, list[Simplex]] = {}
        self._n_delta: dict[int, np.ndarray] = {}
        self._n_rank: dict[int, int] = {}

    def n_faces(self, dim: int) -> list[Simplex]:
        """Return the simplices of the given dimension."""
        if dim < -1:
            raise ValueError("Dimension must be at least -1")
        if dim == -1 or dim > self._sc.dim:
            return []

        if dim not in self._n_faces:
            self._n_faces[dim] = sorted(
                self._sc.n_faces(dim),
                key=lambda simplex: simplex.vertices,
            )

        return self._n_faces[dim]

    def boundary_matrix(self, dim: int) -> np.ndarray:
        """Return the mod-2 boundary matrix from dimension dim to dim - 1."""
        if dim not in self._n_delta:
            row_faces = self.n_faces(dim - 1)
            col_faces = self.n_faces(dim)

            mat = np.zeros((len(row_faces), len(col_faces)), dtype=np.uint8)

            # Rows are (dim - 1)-faces and columns are dim-faces.
            for i, row_face in enumerate(row_faces):
                for j, col_face in enumerate(col_faces):
                    if row_face.is_face(col_face):
                        mat[i, j] = 1

            self._n_delta[dim] = mat

        return self._n_delta[dim]

    def rank_boundary_matrix(self, dim: int) -> int:
        """Return the rank of the boundary matrix over GF(2)."""
        if dim not in self._n_rank:
            mat = self.boundary_matrix(dim).copy()
            rows, cols = mat.shape

            rank = min(rows, cols)

            for k in range(min(rows, cols)):
                # Find a pivot in the remaining submatrix.
                nz = np.argwhere(mat[k:, k:] != 0)

                if nz.size == 0:
                    rank = k
                    break

                # Move the pivot to (k, k).
                i, j = nz[0] + k

                mat[[k, i], :] = mat[[i, k], :]
                mat[:, [k, j]] = mat[:, [j, k]]

                # Clear the rest of the pivot row and column.
                for nzcol in np.flatnonzero(mat[k, k+1:]) + k+1:
                    mat[:, nzcol] ^= mat[:, k]

                for nzrow in np.flatnonzero(mat[k+1:, k]) + k+1:
                    mat[nzrow, :] ^= mat[k, :]

            self._n_rank[dim] = rank

        return self._n_rank[dim]

    def smith_boundary_matrix(self, dim: int) -> np.ndarray:
        """Return the Smith normal form over GF(2)."""
        rank = self.rank_boundary_matrix(dim)
        mat = np.zeros(self.boundary_matrix(dim).shape, dtype=np.uint8)

        for k in range(rank):
            mat[k, k] = 1

        return mat

    def boundary_space_dim(self, k: int) -> int:
        """Return the dimension of the boundary space B_k."""
        return self.rank_boundary_matrix(k + 1)

    def cycle_space_dim(self, k: int) -> int:
        """Return the dimension of the cycle space Z_k."""
        return len(self.n_faces(k)) - self.rank_boundary_matrix(k)

    def betti_number(self, k: int) -> int:
        """Return the k-th Betti number over GF(2)."""
        if k < 0:
            raise ValueError("Betti numbers are only defined for k >= 0")

        return self.cycle_space_dim(k) - self.boundary_space_dim(k)

Una vez más, comprobamos que funciona con el tetraedro:

In [64]:
sc = SimplicialComplex([Simplex((0, 1, 2, 3))])
msc = MatrixSimplicialComplex(sc)

for i in range(4):
    print(f"betti({i}) = {msc.betti_number(i)}")

betti(0) = 1
betti(1) = 0
betti(2) = 0
betti(3) = 0


**Exercise 4:** Calcular los números de Betti del tetraedro.

Para facilitar los ejercicios siguientes, se crea la *factory class* `SimplicialComplexFactory`, para instanciar distintos ejemplos de complejos simpliciales.

In [65]:
class SimplicialComplexFactory:
    """Factory for standard simplicial complexes used in examples and tests."""

    @staticmethod
    def n_simplicial(n: int) -> SimplicialComplex:
        if n < 0:
            raise ValueError("Dimension must be non-negative")
            
        """Return the (n - 1)-simplex on the vertices 0 to n - 1."""
        return SimplicialComplex([Simplex(tuple(range(n)))])

    @staticmethod
    def torus() -> SimplicialComplex:
        """Return a triangulation of the torus."""
        return SimplicialComplex(map(Simplex, [
                (1,2,4), (2,4,5), (2,3,5), (3,5,6), (1,3,6), (1,4,6),
                (4,5,7), (5,7,8), (5,6,8), (6,8,9), (4,6,9), (4,7,9),
                (1,7,8), (1,2,8), (2,8,9), (2,3,9), (3,7,9), (1,3,7)
            ]
        ))

    @staticmethod
    def torus_minimal() -> SimplicialComplex:
        """Return a minimal triangulation of the torus."""
        return SimplicialComplex(map(Simplex, [
                (1,3,7), (1,2,3), (2,3,6), (3,4,6), (4,6,7), (1,6,7), (2,5,6),
                (1,5,6), (2,5,7), (3,5,7), (1,4,5), (3,4,5), (1,2,4), (2,4,7)
            ]
        ))

    @staticmethod
    def klein_bottle() -> SimplicialComplex:
        """Return a triangulation of the Klein bottle."""
        return SimplicialComplex(map(Simplex, [
                (1,2,4), (2,4,5), (2,3,5), (3,5,6), (1,3,6), (1,7,6),
                (4,5,7), (5,7,8), (5,6,8), (6,8,9), (4,6,9), (4,7,9),
                (1,7,8), (1,2,8), (2,8,9), (2,3,9), (3,7,9), (1,4,3)
            ]
        ))

    @staticmethod
    def ring() -> SimplicialComplex:
        """Return a triangulation of the annulus."""
        return SimplicialComplex(map(Simplex, [
                (1,2,4),(1,3,6),(1,4,6),
                (2,3,5),(2,4,5),(3,5,6)
            ]
        ))

    @staticmethod
    def projective_plane() -> SimplicialComplex:
        """Return a triangulation of the projective plane."""
        return SimplicialComplex(map(Simplex, [
                (1,2,6), (2,3,4), (1,3,4), (1,2,5), (2,3,5),
                (1,3,6), (2,4,6), (1,4,5), (3,5,6), (4,5,6)
            ]
        ))

    @staticmethod
    def double_torus() -> SimplicialComplex:
        """Return a triangulation of the double torus."""
        return SimplicialComplex(map(Simplex, [
                (0, 1, 3), (0, 1, 5), (0, 2, 3), (0, 2, 6), (0, 4, 5), (0, 4, 9),
                (0, 6, 8), (0, 7, 8), (0, 7, 9), (1, 2, 4), (1, 2, 6), (1, 3, 4),
                (1, 5, 6), (2, 3, 5), (2, 4, 5), (3, 4, 9), (3, 5, 8), (3, 6, 7),
                (3, 6, 9), (3, 7, 8), (5, 6, 7), (5, 7, 9), (5, 8, 9), (6, 8, 9),
            ]
        ))

    @staticmethod
    def dunce_hat() -> SimplicialComplex:
        """Return a triangulation of the dunce hat."""
        return SimplicialComplex(map(Simplex, [
                (1,3,5), (1,5,6), (1,3,6), (2,4,5),
                (4,5,6), (4,6,8), (6,7,8), (3,6,7),
                (3,2,7), (1,2,7), (1,7,8), (1,2,4),
                (1,3,4), (3,4,8), (2,3,8), (1,2,8)
            ]
        ))

    @staticmethod
    def example() -> SimplicialComplex:
        """Return a mixed example complex with simplices of different dimensions."""
        return SimplicialComplex(map(Simplex, [
                (0, 1),
                (1, 2, 3, 4),
                (4, 5), (5, 6), (4, 6),
                (6, 7, 8),
                (8, 9)
            ]
        ))

    @staticmethod
    def alpha_complex_example_1() -> SimplicialComplex:
        """Return the alpha complex of the first point cloud at radius 0.2."""
        r = 0.20
        points = [
            (0.38021546727456423, 0.4641920233959878), (0.7951628297672293, 0.4926363013586947),
            (0.566623772375203, 0.038325621649018426), (0.3369306814864865, 0.7103735061134965),
            (0.08272837815822842, 0.2263273314352896), (0.5180166301873989, 0.6271769943824689),
            (0.33691411899985035, 0.8402045183219995), (0.33244488399729255, 0.452463652047520),
            (0.11778991601260325, 0.6657734204021165), (0.9384303415747769, 0.2313873874340855)
        ]

        alpha_filtration = AlphaFiltration(points).build()

        return alpha_filtration.filtration(r)

    @staticmethod
    def alpha_complex_example_2() -> SimplicialComplex:
        """Return the alpha complex of the first point cloud at radius 1.0."""
        r = 1.00
        points = [
            (0.38021546727456423, 0.4641920233959878), (0.7951628297672293, 0.4926363013586947),
            (0.566623772375203, 0.038325621649018426), (0.3369306814864865, 0.7103735061134965),
            (0.08272837815822842, 0.2263273314352896), (0.5180166301873989, 0.6271769943824689),
            (0.33691411899985035, 0.8402045183219995), (0.33244488399729255, 0.452463652047520),
            (0.11778991601260325, 0.6657734204021165), (0.9384303415747769, 0.2313873874340855)
        ]

        alpha_filtration = AlphaFiltration(points).build()

        return alpha_filtration.filtration(r)

Probamos el tetraedro:

In [66]:
sc = SimplicialComplexFactory.n_simplicial(4)
msc = MatrixSimplicialComplex(sc)

for i in range(sc.dim+1):
    print(f"betti({i}) = {msc.betti_number(i)}")

betti(0) = 1
betti(1) = 0
betti(2) = 0
betti(3) = 0


**Exercise 5:** Calcular los números de Betti del borde del tetraedro.

In [67]:
sc = SimplicialComplex(
    SimplicialComplexFactory.n_simplicial(4).skeleton(2)
)

msc = MatrixSimplicialComplex(sc)

for i in range(sc.dim+1):
    print(f"betti({i}) = {msc.betti_number(i)}")

betti(0) = 1
betti(1) = 0
betti(2) = 1


**Exercise 6:** Calcular los números de Betti del toro con las dos triangulaciones que hemos visto en clase.

Comprobamos primero una triangulación normal:

In [68]:
sc = SimplicialComplexFactory.torus()
msc = MatrixSimplicialComplex(sc)

for i in range(sc.dim+1):
    print(f"betti({i}) = {msc.betti_number(i)}")

betti(0) = 1
betti(1) = 2
betti(2) = 1


Y a continuación probamos la triangulación mínima:

In [69]:
sc = SimplicialComplexFactory.torus_minimal()
msc = MatrixSimplicialComplex(sc)

for i in range(sc.dim+1):
    print(f"betti({i}) = {msc.betti_number(i)}")

betti(0) = 1
betti(1) = 2
betti(2) = 1


**Exercise 7:** Calcular los números de Betti de la botella de Klein.

In [70]:
sc = SimplicialComplexFactory.klein_bottle()
msc = MatrixSimplicialComplex(sc)

for i in range(sc.dim+1):
    print(f"betti({i}) = {msc.betti_number(i)}")

betti(0) = 1
betti(1) = 3
betti(2) = 0


**Exercise 8:** Calcular los números de Betti del anillo.

In [71]:
sc = SimplicialComplexFactory.ring()
msc = MatrixSimplicialComplex(sc)

for i in range(sc.dim+1):
    print(f"betti({i}) = {msc.betti_number(i)}")

betti(0) = 1
betti(1) = 1
betti(2) = 0


**Exercise 9:** Calcular los números de Betti del plano proyectivo.

In [72]:
sc = SimplicialComplexFactory.projective_plane()
msc = MatrixSimplicialComplex(sc)

for i in range(sc.dim+1):
    print(f"betti({i}) = {msc.betti_number(i)}")

betti(0) = 1
betti(1) = 1
betti(2) = 1


**Exercise 10:** Calcular los números de Betti del sombrero del asno.

In [73]:
sc = SimplicialComplexFactory.dunce_hat()
msc = MatrixSimplicialComplex(sc)

for i in range(sc.dim+1):
    print(f"betti({i}) = {msc.betti_number(i)}")

betti(0) = 1
betti(1) = 1
betti(2) = 0


**Exercise 11:** Calcular los números de Betti del doble toro.

In [74]:
sc = SimplicialComplexFactory.double_torus()
msc = MatrixSimplicialComplex(sc)

for i in range(sc.dim+1):
    print(f"betti({i}) = {msc.betti_number(i)}")

betti(0) = 1
betti(1) = 4
betti(2) = 1


**Exercise 12:** Calcular los números de Betti del complejo simplicial de la transparencia 4.

In [75]:
sc = SimplicialComplexFactory.example()
msc = MatrixSimplicialComplex(sc)

for i in range(sc.dim+1):
    print(f"betti({i}) = {msc.betti_number(i)}")

betti(0) = 1
betti(1) = 1
betti(2) = 0
betti(3) = 0


**Exercise 13:** Calcular los números de Betti de algunos alfa complejos.

Comprobamos los números de Betti de un alfa complejo al variar el instante:

In [76]:
sc = SimplicialComplexFactory.alpha_complex_example_1()
msc = MatrixSimplicialComplex(sc)

for i in range(sc.dim+1):
    print(f"betti({i}) = {msc.betti_number(i)}")

betti(0) = 2
betti(1) = 0
betti(2) = 0


In [77]:
sc = SimplicialComplexFactory.alpha_complex_example_2()
msc = MatrixSimplicialComplex(sc)

for i in range(sc.dim+1):
    print(f"betti({i}) = {msc.betti_number(i)}")

betti(0) = 1
betti(1) = 0
betti(2) = 0


**Exercise 14:** Implementa el algoritmo incremental para calcular los 
números de Betti $\beta_0$ y $\beta_1$ de un complejo simplicial contenido en el plano

Si un complejo simplicial $K$ está contenido en el plano, $\nexists \sigma \in K : \operatorname{dim} \sigma > 2$. Por tanto, para todo $p > 2$, se tiene que $C_p(K) = \{\vec{0}\} \Rightarrow H_p(K) = \{\vec{0}\}$ y, por tanto, $\beta_p = 0$. Además, como $|K| \in \mathbb{R}^2$, $\beta_2 = 0$. Solo tendremos que calcular los valores de $\beta_0$ y $\beta_1$.

Recordamos que $\beta_0$ es el número de componentes conexas del grafo dado por $G = \left(\operatorname{Vert} K, K^{(1)} - K^{(0)} \right)$. Además, por el algoritmo incremental, un símplice puede ser:
* Negativo, si disminuye $\beta_0$ en una unidad.
* Positivo, si aumenta $\beta_1$ en una unidad.

Todo símplice debe ser positivo o negativo, pero no ambas cosas a la vez. Por ello, se puede estudiar el comportamiento de los números de Betti en relación con la dimensión del símplice a agregar:
* Si el símplice tiene dimensión $0$, es un vértice y, por tanto, es positivo. Todo vértice aporta $+1$ a $\beta_0$
* Si el símplice tiene dimensión $2$, como $\beta_2 = 0$ siempre, no puede ser positivo y por tanto es negativo. Cada símplice de dimensión $2$ aporta $-1$ a $\beta_1$
* Si el símplice tiene dimensión $1$, es una arista y puede ser positiva o negativa. Para averiguarlo, estudiamos el comportamiento de $\beta_0$, que es el número de componentes conexas del grafo $G$ definido anteriormente.

Una arista puede disminuir en una unidad el número de componentes conexas ($\beta_0$) si une dos vértices que estén en distintas componentes; o puede no variarlo si une dos vértices que ya estaban en la misma componente conexa. En este segundo caso, $\beta_0$ no varía y, por lo anterior, aumentaría en una unidad $\beta_1$. Por todo esto, para averiguar el comportamiento de una arista basta con ver si altera el número de componentes conexas. Para hacer esto de una manera incremental, la manera más sencilla y eficiente es usar una estructura *DSU* (*Disjoint Set Union*) para controlar si dos vértices están en la misma componente conexa.

Esta estructura sirve para, como su nombre indica, realizar uniones de conjuntos disjuntos de manera muy eficiente; modelando cada conjunto disjunto como un árbol. Así, cada conjunto tiene un representante (raíz) que sirve para saber si dos elementos están en distintos conjuntos; y la unión se realiza conectando una raíz con otra. Esta estructura tiene tres operaciones:
1. `make_set(u)`: Se usa para añadir el conjunto unipuntual $\{u\}$ al *DSU*
2. `find_set(u)`: Se usa para buscar el representante del conjunto que contiene a $u$. 
3. `union_set(u,v) -> Bool`: Se usa para unir los conjuntos que contienen a $u$ y $v$. No hace falta que ni $u$ ni $v$ sean los representantes de sus respectivos conjuntos. Devuelve `False` si $u$ y $v$ ya se encontraban en el mismo conjunto (mismo representante); y `True` en caso contrario.

Con esto, y teniendo en cuenta que la operación `union_set` ya comprueba si los dos puntos ya estaban en el mismo conjunto, se tiene un algoritmo sencillo para la positividad/negatividad de un símplice de dimensión $1$.

1. Sea $\sigma = (u, v)$ el $1$-símplice a estudiar
2. Si `union_set(u,v)`:

    -- $u$ y $v$ no estaban en la misma componente conexa y $\sigma$ los ha unido.

    $\beta_0 \leftarrow \beta_0 - 1$
3. En otro caso:
    
    -- $u$ y $v$ ya estaban en la misma componente conexa. Por tanto $\beta_0$ no cambia y tiene que cambiar $\beta_1$

    $\beta_1 \leftarrow \beta_1 + 1$

Incluimos una implementación del *DSU*:

In [78]:
class DSU:
    """Disjoint-set union structure with path compression and union by size."""

    def __init__(self) -> None:
        """Initializes the data structure."""
        self.parent = {}
        self.size = {}

    def make_set(self, n: int):
        """Add the element n as a new singleton set."""
        self.parent[n] = n
        self.size[n] = 1

    def find_set(self, n: int):
        """Returns the representative of the set containing n."""
        repr = self.parent[n]

        if repr == n:
            return n

        new_repr = self.find_set(repr)
        self.parent[n] = new_repr

        return new_repr

    def union_set(self, n1: int, n2: int) -> bool:
        """Merges the sets containing n1 and n2 and returns whether they were already merged."""
        repr1 = self.find_set(n1)
        repr2 = self.find_set(n2)

        if repr1 == repr2:
            return False

        if self.size[repr1] > self.size[repr2]:
            repr1, repr2 = repr2, repr1

        self.parent[repr1] = repr2
        self.size[repr2] += self.size[repr1]

        return True

Y el código completo de la clase `SimplicialComplex`, con el algoritmo anterior añadido en la función `betti_planar`:

In [79]:
from __future__ import annotations

from collections import defaultdict
from collections.abc import Iterable

from homology.simplex import Simplex
from homology.utils.DSU import DSU


class SimplicialComplex:
    """Represents a simplicial complex: a set of simplices closed under taking faces."""

    @staticmethod
    def _euler_characteristic(face_set: set[Simplex]) -> int:
        """Compute the Euler characteristic using face dimensions."""
        chi = 0

        for face in face_set:
            if face.dim() % 2 == 0:
                chi += 1
            else:
                chi -= 1

        return chi

    def __init__(self, generator: Iterable[Simplex]) -> None:
        """Initialize the simplicial complex with a set of simplices."""
        self.dim: int = -1
        self._face_set: set[Simplex] = set()

        for simplex in generator:
            faces = simplex.faces()

            self._face_set.update(faces)

            if simplex.dim() > self.dim:
                self.dim = simplex.dim()

        self.euler_characteristic: int = self._euler_characteristic(self.face_set)

    @property
    def face_set(self) -> set[Simplex]:
        """Return the set of all simplices (treat as read-only)."""
        return self._face_set

    def n_faces(self, dim: int) -> set[Simplex]:
        """Return all simplices with the specified dimension."""
        return {face for face in self.face_set if face.dim() == dim}

    def skeleton(self, dim: int) -> set[Simplex]:
        """Return all simplices with dimension less than or equal to the specified value."""
        return {face for face in self.face_set if face.dim() <= dim}

    def st(self, simplex: Simplex) -> set[Simplex]:
        """Return the star of the simplex."""
        res = {member for member in self.face_set if simplex.is_face(member)}

        if not res:
            raise ValueError("'simplex' is not a face")

        return res

    def closed_st(self, simplex: Simplex) -> SimplicialComplex:
        """Return the closed star of the simplex."""
        return SimplicialComplex(self.st(simplex))

    def lk(self, simplex: Simplex) -> set[Simplex]:
        """Return the link of the simplex."""
        closed = self.closed_st(simplex)
        return {elem for elem in closed.face_set if simplex.is_disjoint(elem)}

    def graph(self) -> dict[int, list[int]]:
        """Return a dictionary representing the 1-skeleton as a graph."""
        graph = defaultdict(list)

        for v1, v2 in self.n_faces(1):
            graph[v1].append(v2)
            graph[v2].append(v1)

        return graph

    def connected_components(self) -> int:
        """Return the number of connected components."""
        visited = set()

        graph = self.graph()
        vertices = [v[0] for v in self.n_faces(0)]

        components = 0

        for u in vertices:
            if u in visited:
                continue

            queue = [u]
            visited.add(u)

            while queue:
                v = queue.pop()

                for neig in graph[v]:
                    if neig not in visited:
                        queue.append(neig)
                        visited.add(neig)

            components += 1

        return components

    def insert(self, generator: Iterable[Simplex]) -> None:
        """Insert a generator of simplices into the complex."""
        additions = set()

        for simplex in generator:
            difference = simplex.faces() - self.face_set

            additions.update(difference)
            self._face_set.update(difference)

            if simplex.dim() > self.dim:
                self.dim = simplex.dim()

        self.euler_characteristic += self._euler_characteristic(additions)

    def betti_planar(self) -> tuple[int, int]:
        """Computes the pair of betti numbers describing a planar simplicial complex."""

        dsu = DSU()

        beta_0 = 0
        beta_1 = 0

        edges = []

        for face in self._face_set:
            match face.dim():
                case 0:
                    beta_0 += 1
                    dsu.make_set(face.vertices[0])
                case 1:
                    edges.append(face.vertices)
                case 2:
                    beta_1 -= 1
                case _:
                    raise ValueError("The algorithm only works if the simplicial complex is planar")
        
        for u, v in edges:
            if dsu.union_set(u, v):
                beta_0 -= 1
            else:
                beta_1 += 1
        
        return (beta_0, beta_1)


Probaremos el algoritmo anterior usando el primer alfa-complejo del **Exercise 13** de esta práctica

In [80]:
sc = SimplicialComplexFactory.alpha_complex_example_1()
beta_0, beta_1 = sc.betti_planar()
print(f"betti(0) = {beta_0}")
print(f"betti(1) = {beta_1}")

betti(0) = 2
betti(1) = 0


Ahora, hacemos lo mismo con el segundo alfa-complejo

In [81]:
sc = SimplicialComplexFactory.alpha_complex_example_2()
beta_0, beta_1 = sc.betti_planar()
print(f"betti(0) = {beta_0}")
print(f"betti(1) = {beta_1}")

betti(0) = 1
betti(1) = 0


Los resultados de ambas pruebas coinciden con los obtenidos usando la matriz del operador borde.